# SmolVLM-256M · Augmentation and inverse-frequency sampling

| | |
|---|---|
| **Project** | EdgeVLM-Drive — resource-aware vision-language driver monitoring on NVIDIA Jetson Nano |
| **Model** | SmolVLM-256M-Instruct |
| **Stage** | 2 of 12 · Augmentation and inverse-frequency sampling |
| **Purpose** | Offline augmentation of rare classes with inverse-frequency sampling; direct evaluation. |
| **Input** | train / validation / test CSVs with `frame_path` and `label` (see `data/README.md`) |
| **Output** | LoRA adapter (`best_by_macro_f1`), predictions CSV, metrics |
| **Configuration** | [`02_augmented_sampling.json`](../../configs/smolvlm256m/02_augmented_sampling.json) |
| **Results** | [`single_image_sampling_direct.json`](../../results/smolvlm_256m/single_image_sampling_direct.json) |
| **Type** | Experiment; cell outputs from the original run are retained |
| **Setup** | set the paths in the first configuration cell to your data and split folders |


# EdgeVLM-Drive SmolVLM fine-tuning with cleaned augmented data

Training uses the deduplicated augmented CSV. Validation and test remain from the original splits. Run all cells from top to bottom after restarting the kernel.

In [1]:
import os
from pathlib import Path

# Keep optional frameworks from being imported by Transformers later.
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_FLAX"] = "0"

import pandas as pd
from PIL import Image

PROJECT_ROOT = Path("/home/jovyan/EdgeVLM_finetune")
FRAME_ROOT = Path("/home/jovyan")
SPLITS_DIR = Path("/home/jovyan/splits")
AUGMENTED_SPLITS_DIR = Path("/home/jovyan/splits_augmented_v2")

# Only the training split is replaced by the augmented data.
TRAIN_CSV = AUGMENTED_SPLITS_DIR / "train_augmented_deduplicated.csv"
VAL_CSV = SPLITS_DIR / "val.csv"
TEST_CSV = SPLITS_DIR / "test.csv"

LABELS = [
    "NORMAL",
    "DROWSY",
    "LOOKING_AWAY",
    "PHONE_USAGE",
]

print("Notebook directory:", Path.cwd())
print("Augmented directory:", AUGMENTED_SPLITS_DIR)
print("Training CSV:", TRAIN_CSV)
print("Validation CSV:", VAL_CSV)
print("Test CSV:", TEST_CSV)

for path in [TRAIN_CSV, VAL_CSV, TEST_CSV]:
    print(path, "exists:", path.is_file())


Notebook directory: /home/jovyan/EdgeVLM_finetune
Augmented directory: /home/jovyan/splits_augmented_v2
Training CSV: /home/jovyan/splits_augmented_v2/train_augmented_deduplicated.csv
Validation CSV: /home/jovyan/splits/val.csv
Test CSV: /home/jovyan/splits/test.csv
/home/jovyan/splits_augmented_v2/train_augmented_deduplicated.csv exists: True
/home/jovyan/splits/val.csv exists: True
/home/jovyan/splits/test.csv exists: True


## 1. Read the three CSV files

In [2]:
train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)

datasets = {
    "train": train_df,
    "validation": val_df,
    "test": test_df,
}

required_columns = {
    "frame_path",
    "label",
    "source_dataset",
    "group",
    "subject",
}

for name, dataframe in datasets.items():
    missing_columns = required_columns - set(dataframe.columns)
    if missing_columns:
        raise ValueError(f"{name} is missing columns: {missing_columns}")

    unknown_labels = set(dataframe["label"].astype(str)) - set(LABELS)
    if unknown_labels:
        raise ValueError(f"{name} has unknown labels: {unknown_labels}")

    print(f"{name}: {len(dataframe)} rows")
    print(dataframe["label"].value_counts().reindex(LABELS, fill_value=0))
    print()

print("First augmented training path:")
print(train_df.iloc[0]["frame_path"])


train: 17893 rows
label
NORMAL          9390
DROWSY          1705
LOOKING_AWAY    2233
PHONE_USAGE     4565
Name: count, dtype: int64

validation: 3941 rows
label
NORMAL          2470
DROWSY           289
LOOKING_AWAY     310
PHONE_USAGE      872
Name: count, dtype: int64

test: 3579 rows
label
NORMAL          2151
DROWSY           168
LOOKING_AWAY     302
PHONE_USAGE      958
Name: count, dtype: int64

First augmented training path:


## 2. Resolve augmented and original image paths

In [3]:
# The augmented CSV may store either a complete path or a short path
# such as gb_face/frame_00000.jpg. This resolver supports both forms.

AUGMENTED_IMAGE_DIRS = {
    "DROWSY": AUGMENTED_SPLITS_DIR / "augmented_drowsy_images",
    "LOOKING_AWAY": AUGMENTED_SPLITS_DIR / "augmented_looking_away_images",
}


def build_augmented_suffix_index():
    index = {}

    if not AUGMENTED_SPLITS_DIR.exists():
        return index

    for image_path in AUGMENTED_SPLITS_DIR.rglob("*"):
        if not image_path.is_file():
            continue
        if image_path.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
            continue

        relative_parts = image_path.relative_to(AUGMENTED_SPLITS_DIR).as_posix().split("/")
        first_start = max(0, len(relative_parts) - 6)

        for start in range(first_start, len(relative_parts)):
            key = "/".join(relative_parts[start:])
            index.setdefault(key, []).append(image_path)

    return index


_AUGMENTED_SUFFIX_INDEX = build_augmented_suffix_index()


def resolve_frame_path(frame_path, split_name, label=None):
    path_text = str(frame_path).strip().replace("\\", "/")
    relative_path = Path(path_text)

    if relative_path.is_absolute() and relative_path.is_file():
        return relative_path

    label_text = str(label).strip().upper()
    preferred_dir = AUGMENTED_IMAGE_DIRS.get(label_text)

    if split_name == "train":
        candidate_roots = []
        if preferred_dir is not None:
            candidate_roots.append(preferred_dir)
        candidate_roots.extend([AUGMENTED_SPLITS_DIR, FRAME_ROOT])
    else:
        candidate_roots = [FRAME_ROOT]

    for root in candidate_roots:
        candidate = root / relative_path
        if candidate.is_file():
            return candidate

    key = relative_path.as_posix().lstrip("./")
    matches = list(_AUGMENTED_SUFFIX_INDEX.get(key, []))

    if preferred_dir is not None:
        preferred_matches = [
            item for item in matches
            if preferred_dir in item.parents
        ]
        if preferred_matches:
            return preferred_matches[0]

    if len(matches) == 1:
        return matches[0]

    if len(matches) > 1:
        raise FileNotFoundError(
            f"Ambiguous path {frame_path} for label {label}: {matches[:5]}"
        )

    raise FileNotFoundError(
        f"Could not resolve {frame_path} for split {split_name}"
    )


print("Indexed augmented image entries:", len(_AUGMENTED_SUFFIX_INDEX))


Indexed augmented image entries: 3112


## 3. Check image availability and split integrity

In [4]:
resolved_paths = {}

for name, dataframe in datasets.items():
    resolved = []
    missing = []

    for _, row in dataframe.iterrows():
        try:
            resolved.append(
                resolve_frame_path(
                    row["frame_path"],
                    split_name=name,
                    label=row["label"],
                )
            )
        except FileNotFoundError as error:
            resolved.append(None)
            missing.append(str(error))

    resolved_paths[name] = resolved

    print(f"{name} missing images: {len(missing)}")
    if missing:
        print("Example:", missing[0])

# Subjects must not occur in more than one split.
for first, second in [
    ("train", "validation"),
    ("train", "test"),
    ("validation", "test"),
]:
    first_subjects = set(datasets[first]["subject"].astype(str))
    second_subjects = set(datasets[second]["subject"].astype(str))
    print(
        "subject overlap",
        first,
        "/",
        second,
        ":",
        len(first_subjects.intersection(second_subjects)),
    )

print(
    "Note: group overlap can exist because groups are broader recording groups; "
    "the important split constraint here is subject disjointness."
)


train missing images: 0
validation missing images: 0
test missing images: 0
subject overlap train / validation : 0
subject overlap train / test : 0
subject overlap validation / test : 0
Note: group overlap can exist because groups are broader recording groups; the important split constraint here is subject disjointness.


## 4. Inspect image format and required preprocessing

In [5]:
# Inspect representative images from each split.
for name, dataframe in datasets.items():
    valid_index = next(
        index for index, path in enumerate(resolved_paths[name])
        if path is not None
    )
    image_path = resolved_paths[name][valid_index]

    with Image.open(image_path) as image_file:
        print(
            name,
            "| path:", image_path,
            "| format:", image_file.format,
            "| mode:", image_file.mode,
            "| size:", image_file.size,
        )

# Check that the files can be opened and converted to RGB.
# This does not resize or normalize the pixels.
for name, paths in resolved_paths.items():
    readable = 0
    rgb_convertible = 0

    for image_path in paths:
        if image_path is None:
            continue

        try:
            with Image.open(image_path) as image_file:
                readable += 1
                image_file.convert("RGB")
                rgb_convertible += 1
        except Exception:
            pass

    print(
        name,
        "| readable:", readable,
        "| RGB-convertible:", rgb_convertible,
        "| total:", len(paths),
    )

print()
print("Preparation decision:")
print("1. Keep the original image content.")
print("2. Convert every image to RGB when it is loaded.")
print("3. Do not manually resize or normalize here.")
print("4. Let the SmolVLM processor apply its model-specific resize and normalization later.")
print("5. Do not augment validation or test images.")


train | readable: 17893 | RGB-convertible: 17893 | total: 17893
validation | readable: 3941 | RGB-convertible: 3941 | total: 3941
test | readable: 3579 | RGB-convertible: 3579 | total: 3579

Preparation decision:
1. Keep the original image content.
2. Convert every image to RGB when it is loaded.
3. Do not manually resize or normalize here.
4. Let the SmolVLM processor apply its model-specific resize and normalization later.
5. Do not augment validation or test images.


## Preparation status

If all missing-image counts are zero and all images are RGB-convertible, the data is ready for the next notebook stage. Fine-tuning will be added only after these checks pass.

In [6]:
for name, paths in resolved_paths.items():
    augmented_count = sum(
        "augmented_" in str(path)
        for path in paths
        if path is not None
    )

    duplicate_count = len(paths) - len(set(
        str(path) for path in paths if path is not None
    ))

    print(name)
    print("Augmented images:", augmented_count)
    print("Duplicate resolved paths:", duplicate_count)

train
Augmented images: 1691
Duplicate resolved paths: 0
validation
Augmented images: 0
Duplicate resolved paths: 0
test
Augmented images: 0
Duplicate resolved paths: 0


In [7]:
train_check = train_df.copy()

train_check["resolved_path"] = [
    str(path) if path is not None else None
    for path in resolved_paths["train"]
]

train_check["is_augmented"] = train_check["resolved_path"].str.contains(
    "augmented_",
    case=False,
    na=False,
)

print("Total training rows:", len(train_check))
print(
    "Repeated CSV frame_path rows:",
    train_check["frame_path"].duplicated().sum(),
)
print(
    "Repeated resolved-path rows:",
    train_check["resolved_path"].duplicated().sum(),
)

print("\nMost repeated resolved paths:")
print(
    train_check["resolved_path"]
    .value_counts()
    .head(15)
)

duplicate_rows = train_check[
    train_check["resolved_path"].duplicated(keep=False)
]

print("\nDuplicate rows by label:")
print(duplicate_rows["label"].value_counts())

print("\nDuplicate rows by augmented/original status:")
print(duplicate_rows["is_augmented"].value_counts())

duplicate_summary = (
    train_check
    .groupby("resolved_path")
    .agg(
        rows=("label", "size"),
        different_labels=("label", "nunique"),
        different_csv_paths=("frame_path", "nunique"),
    )
    .query("rows > 1")
    .sort_values("rows", ascending=False)
)

print("\nDuplicate-path summary:")
print(duplicate_summary.head(15))

Total training rows: 17893
Repeated CSV frame_path rows: 0
Repeated resolved-path rows: 0

Most repeated resolved paths:
resolved_path
Name: count, dtype: int64

Duplicate rows by label:
Series([], Name: count, dtype: int64)

Duplicate rows by augmented/original status:
Series([], Name: count, dtype: int64)

Duplicate-path summary:
Empty DataFrame
Columns: [rows, different_labels, different_csv_paths]
Index: []


In [8]:
# Check for conflicting labels before removing duplicates
label_conflicts = (
    train_df.groupby("frame_path")["label"]
    .nunique()
    .loc[lambda values: values > 1]
)

if len(label_conflicts) > 0:
    raise ValueError(
        "Some frame paths have more than one label. Review them before training."
    )

before_count = len(train_df)

# Keep one row for each unique training image
train_df = (
    train_df
    .drop_duplicates(subset=["frame_path"], keep="first")
    .reset_index(drop=True)
)

removed_count = before_count - len(train_df)

print("Rows before cleaning:", before_count)
print("Rows after cleaning:", len(train_df))
print("Duplicate rows removed:", removed_count)

print("\nCleaned training label counts:")
print(train_df["label"].value_counts().reindex(LABELS, fill_value=0))

# Save a clean CSV for the later fine-tuning notebook
CLEAN_TRAIN_CSV = (
    AUGMENTED_SPLITS_DIR / "train_augmented_deduplicated.csv"
)

train_df.to_csv(CLEAN_TRAIN_CSV, index=False)

print("\nSaved cleaned training CSV to:")
print(CLEAN_TRAIN_CSV)

Rows before cleaning: 17893
Rows after cleaning: 17893
Duplicate rows removed: 0

Cleaned training label counts:
label
NORMAL          9390
DROWSY          1705
LOOKING_AWAY    2233
PHONE_USAGE     4565
Name: count, dtype: int64

Saved cleaned training CSV to:
/home/jovyan/splits_augmented_v2/train_augmented_deduplicated.csv


In [9]:
print("Duplicate training paths:",
      train_df["frame_path"].duplicated().sum())

Duplicate training paths: 0


## 5. Install and verify the fine-tuning dependencies

Run this cell once. The model itself is loaded in the next cell.

In [10]:
%pip install -q \
"transformers==4.46.3" \
"tokenizers==0.20.3" \
"peft>=0.13" \
"accelerate>=1.0" \
"datasets>=3.0" \
"safetensors>=0.4" \
"sentencepiece" \
"tqdm" \
"scikit-learn"



[notice] A new release of pip is available: 25.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [11]:
import sys
import torch
import transformers
import tokenizers
import peft
import accelerate

from transformers import (
    AutoProcessor,
    AutoModelForVision2Seq,
    TrainingArguments,
    Trainer,
)

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Transformers:", transformers.__version__)
print("Tokenizers:", tokenizers.__version__)
print("PEFT:", peft.__version__)
print("Accelerate:", accelerate.__version__)


Python: 3.12.13 | packaged by conda-forge | (main, Mar  5 2026, 16:50:00) [GCC 14.3.0]
PyTorch: 2.11.0+cu126
CUDA available: True
GPU: NVIDIA A16
Transformers: 4.46.3
Tokenizers: 0.20.3
PEFT: 0.20.0
Accelerate: 1.14.0


## 6. Load SmolVLM from the existing cache

In [12]:
import gc
import torch
from transformers import AutoProcessor, AutoModelForVision2Seq

MODEL_ID = "HuggingFaceTB/SmolVLM-256M-Instruct"
MODEL_CACHE = PROJECT_ROOT / "model_cache"
MODEL_CACHE.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float16 if device.type == "cuda" else torch.float32

try:
    print("Trying the local model cache...")
    processor = AutoProcessor.from_pretrained(
        MODEL_ID,
        cache_dir=str(MODEL_CACHE),
        local_files_only=True,
    )
    model = AutoModelForVision2Seq.from_pretrained(
        MODEL_ID,
        cache_dir=str(MODEL_CACHE),
        local_files_only=True,
        torch_dtype=dtype,
    )
    print("Loaded from local cache.")
except OSError:
    print("Local cache is incomplete; downloading missing model files...")
    processor = AutoProcessor.from_pretrained(
        MODEL_ID,
        cache_dir=str(MODEL_CACHE),
    )
    model = AutoModelForVision2Seq.from_pretrained(
        MODEL_ID,
        cache_dir=str(MODEL_CACHE),
        torch_dtype=dtype,
    )

model = model.to(device)
model.eval()

print("Model:", MODEL_ID)
print("Device:", device)
print("Parameters:", sum(parameter.numel() for parameter in model.parameters()))


Some kwargs in processor config are unused and will not have any effect: image_seq_len. 


Trying the local model cache...
Loaded from local cache.
Model: HuggingFaceTB/SmolVLM-256M-Instruct
Device: cuda
Parameters: 256484928


## 7. Define the fine-tuning dataset

In [13]:
from torch.utils.data import Dataset
from PIL import Image

INSTRUCTION = (
    "Classify the driver's visible state. "
    "Return exactly one label from: NORMAL, DROWSY, LOOKING_AWAY, or PHONE_USAGE. "
    "Do not provide an explanation or punctuation."
)

class DriverStateDataset(Dataset):
    def __init__(self, dataframe, split_name):
        self.dataframe = dataframe.reset_index(drop=True)
        self.split_name = split_name

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        image_path = resolve_frame_path(
            row["frame_path"],
            split_name=self.split_name,
            label=row["label"],
        )

        with Image.open(image_path) as image_file:
            image = image_file.convert("RGB").copy()

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "text", "text": INSTRUCTION},
                ],
            }
        ]

        return {
            "image": image,
            "messages": messages,
            "target": str(row["label"]).strip(),
            "path": str(image_path),
        }

train_dataset = DriverStateDataset(train_df, split_name="train")
val_dataset = DriverStateDataset(val_df, split_name="validation")

example = train_dataset[0]
print("Training examples:", len(train_dataset))
print("Validation examples:", len(val_dataset))
print("Example target:", example["target"])
print("Example image:", example["path"])


Training examples: 17893
Validation examples: 3941
Example target: NORMAL


## 8. Create the multimodal training collator

In [14]:
class SmolVLMDataCollator:
    def __init__(self, processor, image_size=512):
        self.processor = processor
        self.image_size = image_size

    def __call__(self, features):
        images = [item["image"] for item in features]
        full_texts = []
        prompt_texts = []

        for item in features:
            user_messages = item["messages"]

            full_messages = user_messages + [
                {
                    "role": "assistant",
                    "content": [
                        {"type": "text", "text": item["target"]}
                    ],
                }
            ]

            prompt_texts.append(
                self.processor.apply_chat_template(
                    user_messages,
                    add_generation_prompt=True,
                    tokenize=False,
                )
            )

            full_texts.append(
                self.processor.apply_chat_template(
                    full_messages,
                    add_generation_prompt=False,
                    tokenize=False,
                )
            )

        images_for_processor = [[image] for image in images]

        processor_arguments = {
            "return_tensors": "pt",
            "padding": True,
            "truncation": True,
            "size": {"longest_edge": self.image_size},
            "max_image_size": {"longest_edge": self.image_size},
        }

        batch = self.processor(
            text=full_texts,
            images=images_for_processor,
            **processor_arguments,
        )

        prompt_batch = self.processor(
            text=prompt_texts,
            images=images_for_processor,
            **processor_arguments,
        )

        labels = batch["input_ids"].clone()
        prompt_lengths = prompt_batch["attention_mask"].sum(dim=1)

        # Train only on the assistant label, not on the image/user prompt.
        for row_number, prompt_length in enumerate(prompt_lengths):
            labels[row_number, :int(prompt_length)] = -100

        pad_token_id = self.processor.tokenizer.pad_token_id
        if pad_token_id is not None:
            labels[labels == pad_token_id] = -100

        batch["labels"] = labels
        return batch

collator = SmolVLMDataCollator(processor)
print("Collator created successfully.")


Collator created successfully.


In [15]:
from torch.utils.data import DataLoader

loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=collator,
)

batch = next(iter(loader))

print("Training batch created successfully.")
for key, value in batch.items():
    if hasattr(value, "shape"):
        print(key, tuple(value.shape))
    else:
        print(key, type(value))

print(
    "Target tokens used for loss:",
    int((batch["labels"] != -100).sum().item()),
)


Training batch created successfully.
pixel_values (2, 1, 3, 512, 512)
pixel_attention_mask (2, 1, 512, 512)
input_ids (2, 126)
attention_mask (2, 126)
labels (2, 126)
Target tokens used for loss: 10


## 9. Attach LoRA adapters to the language component

In [16]:
from peft import LoraConfig, TaskType, get_peft_model, PeftModel

if isinstance(model, PeftModel):
    print("A LoRA adapter is already attached.")
else:
    text_attention_targets = []

    for name, module in model.named_modules():
        if (
            name.startswith("model.text_model.")
            and isinstance(module, torch.nn.Linear)
            and name.endswith(("q_proj", "k_proj", "v_proj", "o_proj"))
        ):
            text_attention_targets.append(name)

    text_attention_targets = sorted(text_attention_targets)

    if not text_attention_targets:
        raise RuntimeError(
            "No text-model attention layers were found. "
            "Inspect model.named_modules() before continuing."
        )

    print("LoRA target layers:", len(text_attention_targets))

    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
        target_modules=text_attention_targets,
    )

    model.config.use_cache = False
    model = get_peft_model(model, lora_config)

model.train()
model.print_trainable_parameters()


LoRA target layers: 120
trainable params: 921,600 || all params: 257,406,528 || trainable%: 0.3580


## 10. Optional LoRA smoke test

This is disabled by default because the wiring was already tested. Set RUN_SMOKE_TEST to True only if you want to test the new data path before full training.

In [17]:
RUN_SMOKE_TEST = False

if RUN_SMOKE_TEST:
    smoke_output = PROJECT_ROOT / "lora_smoke_test_augmented_deduplicated"
    smoke_output.mkdir(parents=True, exist_ok=True)

    smoke_args = TrainingArguments(
        output_dir=str(smoke_output),
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        max_steps=3,
        learning_rate=1e-4,
        fp16=torch.cuda.is_available(),
        logging_steps=1,
        save_strategy="no",
        eval_strategy="no",
        report_to="none",
        remove_unused_columns=False,
        dataloader_num_workers=0,
        optim="adamw_torch",
    )

    smoke_trainer = Trainer(
        model=model,
        args=smoke_args,
        train_dataset=train_dataset,
        data_collator=collator,
    )
  
    print("Starting augmented-data smoke test...")
    smoke_result = smoke_trainer.train()
    print("Smoke test completed.")
    print(smoke_result)
else:
    print("Smoke test skipped.")


Smoke test skipped.


## 11. Fine-tune on the cleaned augmented training set

In [20]:
from collections import Counter
from torch.utils.data import WeightedRandomSampler
from transformers import TrainingArguments, Trainer

class BalancedTrainer(Trainer):
    def _get_train_sampler(self):
        labels = self.train_dataset.dataframe["label"].astype(str).tolist()
        counts = Counter(labels)

        sample_weights = torch.tensor(
            [1.0 / counts[label] for label in labels],
            dtype=torch.double,
        )

        return WeightedRandomSampler(
            weights=sample_weights,
            num_samples=len(sample_weights),
            replacement=True,
        )

FINAL_OUTPUT = PROJECT_ROOT / "smolvlm_256m_lora_augmented_deduplicated"
FINAL_OUTPUT.mkdir(parents=True, exist_ok=True)

training_args = TrainingArguments(
    output_dir=str(FINAL_OUTPUT),
    overwrite_output_dir=True,

    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,

    learning_rate=1e-4,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",

    fp16=torch.cuda.is_available(),

    logging_steps=25,
    eval_strategy="steps",
    eval_steps=250,

    save_strategy="steps",
    save_steps=250,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    remove_unused_columns=False,
    dataloader_num_workers=0,
    optim="adamw_torch",
    report_to="none",
)

trainer = BalancedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collator,
)

print("Starting fine-tuning on the cleaned augmented data...")
train_result = trainer.train()

print("Fine-tuning completed.")
print(train_result)

trainer.save_model(str(FINAL_OUTPUT))
processor.save_pretrained(str(FINAL_OUTPUT))

print("Saved adapter and processor to:", FINAL_OUTPUT)


Starting fine-tuning on the cleaned augmented data...


<IPython.core.display.HTML object>

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



Fine-tuning completed.
TrainOutput(global_step=2236, training_loss=0.08380739214915069, metrics={'train_runtime': 9469.4532, 'train_samples_per_second': 1.89, 'train_steps_per_second': 0.236, 'total_flos': 3129592314562560.0, 'train_loss': 0.08380739214915069, 'epoch': 0.9997205611132846})
Saved adapter and processor to: /home/jovyan/EdgeVLM_finetune/smolvlm_256m_lora_augmented_deduplicated


## 12. Evaluate the fine-tuned model on the untouched test set

In [21]:
import time
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)

model.eval()

LABEL_ORDER = [
    "NORMAL",
    "DROWSY",
    "LOOKING_AWAY",
    "PHONE_USAGE",
]

UNSAFE_LABELS = {
    "DROWSY",
    "LOOKING_AWAY",
    "PHONE_USAGE",
}

def parse_prediction(raw_text):
    raw_text = str(raw_text).strip()
    normalized = raw_text.upper()

    exact_valid = normalized in LABEL_ORDER

    cleaned = normalized.strip(" .,;:!?'\"")
    if cleaned in LABEL_ORDER:
        return cleaned, exact_valid

    return "INVALID", False


def predict_one_test_row(row):
    image_path = resolve_frame_path(
        row["frame_path"],
        split_name="test",
        label=row["label"],
    )

    with Image.open(image_path) as image_file:
        image = image_file.convert("RGB").copy()

    prompt = (
        "Classify the driver's visible state as exactly one label: "
        "NORMAL, DROWSY, LOOKING_AWAY, or PHONE_USAGE. "
        "Return only one label without explanation or punctuation."
    )

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": prompt},
            ],
        }
    ]

    chat_prompt = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False,
    )

    inputs = processor(
        text=chat_prompt,
        images=[image],
        return_tensors="pt",
        size={"longest_edge": 512},
        max_image_size={"longest_edge": 512},
    )

    inputs = inputs.to(next(model.parameters()).device)

    start = time.perf_counter()

    with torch.inference_mode():
        generated = model.generate(
            **inputs,
            max_new_tokens=12,
            do_sample=False,
        )

    latency_ms = (time.perf_counter() - start) * 1000

    input_length = inputs["input_ids"].shape[-1]
    generated_only = generated[:, input_length:]

    raw_output = processor.batch_decode(
        generated_only,
        skip_special_tokens=True,
    )[0].strip()

    prediction, exact_valid = parse_prediction(raw_output)

    return raw_output, prediction, exact_valid, latency_ms, str(image_path)


true_labels = []
predicted_labels = []
raw_outputs = []
exact_valid_flags = []
latencies = []
resolved_test_paths = []

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

for _, row in tqdm(
    test_df.iterrows(),
    total=len(test_df),
    desc="Evaluating untouched test images",
):
    raw_output, prediction, exact_valid, latency_ms, resolved_path = (
        predict_one_test_row(row)
    )

    true_labels.append(str(row["label"]).strip().upper())
    predicted_labels.append(prediction)
    raw_outputs.append(raw_output)
    exact_valid_flags.append(exact_valid)
    latencies.append(latency_ms)
    resolved_test_paths.append(resolved_path)

true_labels = np.array(true_labels)
predicted_labels = np.array(predicted_labels)
exact_valid_flags = np.array(exact_valid_flags)
latencies = np.array(latencies)

precision, recall, f1, support = precision_recall_fscore_support(
    true_labels,
    predicted_labels,
    labels=LABEL_ORDER,
    zero_division=0,
)

balanced_accuracy_four_class = recall.mean()

unsafe_mask = np.isin(true_labels, list(UNSAFE_LABELS))
normal_mask = true_labels == "NORMAL"

unsafe_miss_rate = np.mean(
    predicted_labels[unsafe_mask] != true_labels[unsafe_mask]
)

false_alarm_rate = np.mean(
    np.isin(predicted_labels[normal_mask], list(UNSAFE_LABELS))
)

parser_failure_rate = np.mean(predicted_labels == "INVALID")
exact_label_validity_rate = exact_valid_flags.mean()

matrix_labels = LABEL_ORDER + ["INVALID"]
cm = confusion_matrix(
    true_labels,
    predicted_labels,
    labels=matrix_labels,
)

results_df = test_df.copy()
results_df["resolved_path"] = resolved_test_paths
results_df["raw_output"] = raw_outputs
results_df["prediction"] = predicted_labels
results_df["exact_label_valid"] = exact_valid_flags
results_df["latency_ms"] = latencies

results_path = PROJECT_ROOT / "finetuned_augmented_test_predictions.csv"
results_df.to_csv(results_path, index=False)

print("\n========== FINAL TEST RESULTS ==========")
print("Evaluation images:", len(test_df))
print("Exact-label validity rate:", round(exact_label_validity_rate * 100, 2), "%")
print("Parser failure rate:", round(parser_failure_rate * 100, 2), "%")
print("Overall accuracy:", round(accuracy_score(true_labels, predicted_labels) * 100, 2), "%")
print("Balanced accuracy:", round(balanced_accuracy_four_class * 100, 2), "%")
print(
    "Macro-F1:",
    round(
        f1_score(
            true_labels,
            predicted_labels,
            labels=LABEL_ORDER,
            average="macro",
            zero_division=0,
        ) * 100,
        2,
    ),
    "%",
)

print("\nPer-class results:")
for label, p, r, f, n in zip(
    LABEL_ORDER,
    precision,
    recall,
    f1,
    support,
):
    print(
        f"{label}: precision={p:.3f}, "
        f"recall={r:.3f}, F1={f:.3f}, samples={n}"
    )

print("\nConfusion matrix:")
print(
    pd.DataFrame(
        cm,
        index=matrix_labels,
        columns=matrix_labels,
    )
)

print("Unsafe-state miss rate:", round(unsafe_miss_rate * 100, 2), "%")
print("False-alarm rate:", round(false_alarm_rate * 100, 2), "%")
print("Mean latency (ms):", round(float(np.mean(latencies)), 2))
print("P95 latency (ms):", round(float(np.percentile(latencies, 95)), 2))

if torch.cuda.is_available():
    peak_memory_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
    print("Peak GPU memory (MB):", round(peak_memory_mb, 2))

print("Predictions saved to:", results_path)


Evaluating untouched test images:   0%|          | 0/3579 [00:00<?, ?it/s]


========== FINAL TEST RESULTS ==========
Evaluation images: 3579
Exact-label validity rate: 99.97 %
Parser failure rate: 0.0 %
Overall accuracy: 80.3 %
Balanced accuracy: 54.79 %
Macro-F1: 56.71 %

Per-class results:
NORMAL: precision=0.793, recall=0.938, F1=0.859, samples=2151
DROWSY: precision=0.900, recall=0.054, F1=0.101, samples=168
LOOKING_AWAY: precision=0.484, recall=0.460, F1=0.472, samples=302
PHONE_USAGE: precision=0.961, recall=0.740, F1=0.836, samples=958

Confusion matrix:
              NORMAL  DROWSY  LOOKING_AWAY  PHONE_USAGE  INVALID
NORMAL          2017       1           106           27        0
DROWSY           158       9             1            0        0
LOOKING_AWAY     161       0           139            2        0
PHONE_USAGE      208       0            41          709        0
INVALID            0       0             0            0        0
Unsafe-state miss rate: 39.99 %
False-alarm rate: 6.23 %
Mean latency (ms): 229.12
P95 latency (ms): 353.65
Peak GPU 